In [2]:
import os,random,glob,zipfile
import numpy as np
import librosa,soundfile as sf
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.nn.functional as F


In [3]:
#PARAMETERS
SR=16000
DURATION=5.0
TARGET_LEN=int(SR*DURATION)
N_FFT=512
HOP_LENGTH=256
BATCH_SIZE=8
NUM_EPOCHS=50
LR=5e-5
WEIGHT_DECAY=1e-5
PATIENCE=10
USE_AMP=True
def unzip_all(zip_folder, extract_to):
    for file in os.listdir(zip_folder):
        if file.endswith(".zip"):
            path = os.path.join(zip_folder, file)
            print("Extracting:", file)
            with zipfile.ZipFile(path, 'r') as z:
                z.extractall(extract_to)
unzip_all(r"C:\Users\JOSHITAA\Documents\404FunNotFound\AI-Voice-Enhancer\data\noisy datset", r"C:\Users\JOSHITAA\Documents\404FunNotFound\AI-Voice-Enhancer\data\noisy datset")
CLEAN_DATASET_DIR = r"C:\Users\JOSHITAA\Documents\404FunNotFound\AI-Voice-Enhancer\data\clean datset"
NOISE_DATASET_DIR = r"C:\Users\JOSHITAA\Documents\404FunNotFound\AI-Voice-Enhancer\data\noisy datset"
mic_inner = os.path.join(CLEAN_DATASET_DIR, "MIC")
CLEAN_ROOT = mic_inner if os.path.exists(mic_inner) else CLEAN_DATASET_DIR
speaker_folders=sorted([
    d for d in os.listdir(CLEAN_ROOT)
    if os.path.isdir(os.path.join(CLEAN_ROOT, d))
])
print("Speakers found:", speaker_folders)
random.seed(42)
random.shuffle(speaker_folders)
split=int(0.8*len(speaker_folders))
train_speakers=speaker_folders[:split]
val_speakers=speaker_folders[split:]

train_clean_files,val_clean_files=[],[]
for spk in speaker_folders:
    path=os.path.join(CLEAN_ROOT,spk)
    wavs=sorted(glob.glob(os.path.join(path,"*.wav")))
    if spk in train_speakers:
        train_clean_files.extend(wavs)
    else:
        val_clean_files.extend(wavs)
noise_files = sorted(
    glob.glob(os.path.join(NOISE_DATASET_DIR, "**", "*.wav"), recursive=True)
)
assert len(train_clean_files)>0, "No training clean files found"
assert len(val_clean_files)>0, "validation clean set is empty"
assert len(noise_files)>0, "No noise files found"
print("Noise files found:", len(noise_files))
print("Example noise file:", noise_files[:5])

Extracting: DKITCHEN_16k.zip
Extracting: DLIVING_16k.zip
Extracting: DWASHING_16k.zip
Extracting: NFIELD_16k.zip
Extracting: NPARK_16k.zip
Extracting: NRIVER_16k.zip
Extracting: OHALLWAY_16k.zip
Extracting: OMEETING_16k.zip
Extracting: PRESTO_16k.zip
Extracting: PSTATION_16k.zip
Speakers found: ['F01', 'F02', 'M01', 'M02']
Noise files found: 160
Example noise file: ['C:\\Users\\JOSHITAA\\Documents\\404FunNotFound\\AI-Voice-Enhancer\\data\\noisy datset\\DKITCHEN\\ch01.wav', 'C:\\Users\\JOSHITAA\\Documents\\404FunNotFound\\AI-Voice-Enhancer\\data\\noisy datset\\DKITCHEN\\ch02.wav', 'C:\\Users\\JOSHITAA\\Documents\\404FunNotFound\\AI-Voice-Enhancer\\data\\noisy datset\\DKITCHEN\\ch03.wav', 'C:\\Users\\JOSHITAA\\Documents\\404FunNotFound\\AI-Voice-Enhancer\\data\\noisy datset\\DKITCHEN\\ch04.wav', 'C:\\Users\\JOSHITAA\\Documents\\404FunNotFound\\AI-Voice-Enhancer\\data\\noisy datset\\DKITCHEN\\ch05.wav']


In [5]:
#MIXING FUNCTION
def mix_with_random_noise(clean, noise_files,sr=SR,target_len=TARGET_LEN):
    clean=librosa.util.fix_length(clean,size=target_len)

    noise_path=random.choice(noise_files)
    noise,_=librosa.load(noise_path,sr=sr)

    if len(noise)>target_len:
        start=random.radint(0,len(noise)-target_len)
        noise=noise[start:start+target_len]
    else:
        noise=np.pad(noise,(0,target_len-len(noise)))

    if random.random()<0.25:
        try:
            speed = random.uniform(0.9, 1.1)
            noise = librosa.effects.time_stretch(noise, rate=speed)
            noise = librosa.util.fix_length(noise, size=target_len)
        except Exception:
            pass


    snr_db = random.uniform(0, 20)
    rms_clean = np.sqrt(np.mean(clean**2) + 1e-12)
    rms_noise = np.sqrt(np.mean(noise**2) + 1e-12)
    desired_rms_noise = rms_clean / (10 ** (snr_db / 20))
    noise = noise * (desired_rms_noise / (rms_noise + 1e-12))
    noise = noise * random.uniform(0.7, 1.2)

    mixed = clean + noise

    maxv = np.max(np.abs(mixed)) + 1e-12
    if maxv > 1.0:
        mixed = mixed / maxv
    return mixed.astype(np.float32), clean.astype(np.float32)

